# KMU-FED Baseline Training

This notebook trains a lightweight SqueezeNet 1.1 model on the cropped KMU-FED training set.

The model is evaluated on the cropped KMU-FED test set to establish the baseline performance before cross-dataset testing.

Pipeline:

1. Load cropped KMU-FED train and test manifests
2. Create PyTorch Dataset and DataLoader
3. Load pretrained SqueezeNet 1.1
4. Modify the final classifier for six emotion classes
5. Train on KMU-FED cropped train images
6. Evaluate on KMU-FED cropped test images
7. Save the trained model

In [ ]:
# Google Drive
from google.colab import drive

# File and path handling
from pathlib import Path
import os
import random

# Data handling
import pandas as pd
import numpy as np

# Image handling
from PIL import Image, ImageOps

# Visualization
import matplotlib.pyplot as plt

# Progress bar
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Torchvision
from torchvision import transforms, models
from torchvision.models import SqueezeNet1_1_Weights

# Metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
drive.mount("/content/gdrive")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

## Project Paths

This section defines the paths to the cropped manifest files and the output directories for saved models and results.

In [ ]:
PROJECT_DIR = Path("/content/gdrive/MyDrive/FER_Generalization_Project")

CROPPED_MANIFEST_DIR = PROJECT_DIR / "data_processed" / "manifests_cropped"

MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results" / "kmu_baseline"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Project directory exists:", PROJECT_DIR.exists())

print("\nCropped manifest directory:", CROPPED_MANIFEST_DIR)
print("Cropped manifest directory exists:", CROPPED_MANIFEST_DIR.exists())

print("\nModel directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
print("Project folder contents:")

if PROJECT_DIR.exists():
    for item in PROJECT_DIR.iterdir():
        print(item)
else:
    print("Project folder not found. Check Google Drive mount or folder name.")

In [ ]:
print("Cropped manifest files:")

if CROPPED_MANIFEST_DIR.exists():
    for item in CROPPED_MANIFEST_DIR.iterdir():
        print(item)
else:
    print("Cropped manifest folder not found.")

## Load Cropped KMU-FED Manifests

The cropped manifest CSV files contain paths to the cropped-face images created in Notebook 02.

In [ ]:
kmu_train_df = pd.read_csv(CROPPED_MANIFEST_DIR / "kmu_train_cropped.csv")
kmu_test_df = pd.read_csv(CROPPED_MANIFEST_DIR / "kmu_test_cropped.csv")

print("KMU train:", kmu_train_df.shape)
print("KMU test:", kmu_test_df.shape)

display(kmu_train_df.head())

In [ ]:
def fix_drive_paths(df):
    """
    Updates saved Colab paths from /content/drive/MyDrive to /content/gdrive/MyDrive.
    This is needed because the preprocessing notebook used /content/drive,
    but this notebook mounts Drive at /content/gdrive.
    """
    old_prefix = "/content/drive/MyDrive"
    new_prefix = "/content/gdrive/MyDrive"

    path_columns = ["path", "original_path", "cropped_path"]

    for col in path_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(
                old_prefix,
                new_prefix,
                regex=False
            )

    return df


kmu_train_df = fix_drive_paths(kmu_train_df)
kmu_test_df = fix_drive_paths(kmu_test_df)

display(kmu_train_df.head())

In [ ]:
def check_image_paths(df, path_col="cropped_path"):
    existing_count = df[path_col].apply(lambda p: Path(p).exists()).sum()
    total_count = len(df)

    print(f"Existing images: {existing_count}/{total_count}")

    if existing_count < total_count:
        missing_df = df[~df[path_col].apply(lambda p: Path(p).exists())]
        print("\nExample missing paths:")
        display(missing_df[[path_col, "label"]].head())


print("KMU train image path check:")
check_image_paths(kmu_train_df)

print("\nKMU test image path check:")
check_image_paths(kmu_test_df)

In [ ]:
print("Train label counts:")
print(kmu_train_df["label"].value_counts())

print("\nTest label counts:")
print(kmu_test_df["label"].value_counts())

print("\nTrain crop status:")
print(kmu_train_df["face_crop_status"].value_counts())

print("\nTest crop status:")
print(kmu_test_df["face_crop_status"].value_counts())

## Define Emotion Classes

The project uses six shared emotion classes across KMU-FED, KDEF, and FER2013.

In [ ]:
class_names = [
    "anger",
    "disgust",
    "fear",
    "happy",
    "sadness",
    "surprise"
]

num_classes = len(class_names)

label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print("Number of classes:", num_classes)
print("Label mapping:", label_to_idx)

## Image Transforms

The model expects input images of size 224×224.

Training images use light augmentation. Test images only use resizing and normalization.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## PyTorch Dataset

This custom Dataset reads cropped-face image paths from the manifest CSV file and returns image tensors with emotion labels.

In [ ]:
class FERManifestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["cropped_path"]
        label = int(row["label_idx"])

        image = Image.open(image_path)
        image = ImageOps.exif_transpose(image)
        image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [ ]:
batch_size = 32

train_dataset = FERManifestDataset(kmu_train_df, transform=train_transform)
test_dataset = FERManifestDataset(kmu_test_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))
print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

## Build SqueezeNet 1.1 Model

This section loads a pretrained SqueezeNet 1.1 model and modifies the final classifier layer for six facial emotion classes.

In [ ]:
def create_squeezenet_model(num_classes):
    """
    Loads pretrained SqueezeNet 1.1 and modifies the final classifier
    for the required number of emotion classes.
    """

    weights = SqueezeNet1_1_Weights.DEFAULT
    model = models.squeezenet1_1(weights=weights)

    # Replace final classifier layer
    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )

    model.num_classes = num_classes

    return model


model = create_squeezenet_model(num_classes)
model = model.to(device)

print(model)

## Loss Function and Optimizer

CrossEntropyLoss is used for multi-class emotion classification.

Adam optimizer is used to update the model weights.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

## Training and Evaluation Functions

The training function updates the model weights using KMU-FED training data.

The evaluation function calculates loss, accuracy, predictions, and true labels without updating the model.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct_predictions += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)

            correct_predictions += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy, all_preds, all_labels

## Train SqueezeNet on KMU-FED

The model is trained only on the cropped KMU-FED training set.

The KMU-FED test set is used to measure baseline performance.

In [ ]:
num_epochs = 20

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": []
}

best_test_accuracy = 0.0
best_model_path = MODEL_DIR / "squeezenet_kmu_fed_best.pth"

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    test_loss, test_acc, test_preds, test_labels = evaluate_model(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device
    )

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}")

    # Save best model
    if test_acc > best_test_accuracy:
        best_test_accuracy = test_acc

        torch.save({
            "model_state_dict": model.state_dict(),
            "num_classes": num_classes,
            "class_names": class_names,
            "label_to_idx": label_to_idx,
            "idx_to_label": idx_to_label,
            "best_test_accuracy": best_test_accuracy
        }, best_model_path)

        print(f"Best model saved with test accuracy: {best_test_accuracy:.4f}")

print("\nTraining complete.")
print("Best test accuracy:", best_test_accuracy)
print("Best model path:", best_model_path)

In [ ]:
history_df = pd.DataFrame(history)

history_csv_path = RESULTS_DIR / "kmu_baseline_training_history.csv"
history_df.to_csv(history_csv_path, index=False)

display(history_df)

print("Training history saved to:", history_csv_path)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_accuracy"], label="Train Accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("KMU-FED Baseline Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Final Evaluation on KMU-FED Test Set

This section loads the best saved SqueezeNet model and evaluates it on the KMU-FED test set.

The final outputs include:

- test accuracy
- classification report
- confusion matrix
- saved result files

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)

best_model = create_squeezenet_model(num_classes)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model = best_model.to(device)

print("Loaded best model from:", best_model_path)
print("Best saved test accuracy:", checkpoint["best_test_accuracy"])

In [ ]:
final_test_loss, final_test_acc, final_preds, final_labels = evaluate_model(
    model=best_model,
    dataloader=test_loader,
    criterion=criterion,
    device=device
)

print("Final KMU-FED Test Loss:", round(final_test_loss, 4))
print("Final KMU-FED Test Accuracy:", round(final_test_acc, 4))

In [ ]:
report = classification_report(
    final_labels,
    final_preds,
    target_names=class_names,
    digits=4
)

print(report)

report_path = RESULTS_DIR / "kmu_baseline_classification_report.txt"

with open(report_path, "w") as f:
    f.write(report)

print("Classification report saved to:", report_path)

In [ ]:
report = classification_report(
    final_labels,
    final_preds,
    target_names=class_names,
    digits=4
)

print(report)

report_path = RESULTS_DIR / "kmu_baseline_classification_report.txt"

with open(report_path, "w") as f:
    f.write(report)

print("Classification report saved to:", report_path)

In [ ]:
report = classification_report(
    final_labels,
    final_preds,
    target_names=class_names,
    digits=4
)

print(report)

report_path = RESULTS_DIR / "kmu_baseline_classification_report.txt"

with open(report_path, "w") as f:
    f.write(report)

print("Classification report saved to:", report_path)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    precision_recall_fscore_support
)

In [ ]:
overall_accuracy = accuracy_score(final_labels, final_preds)

overall_precision = precision_score(
    final_labels,
    final_preds,
    average="macro",
    zero_division=0
)

overall_recall = recall_score(
    final_labels,
    final_preds,
    average="macro",
    zero_division=0
)

overall_metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [overall_accuracy, overall_precision, overall_recall]
})

display(overall_metrics_df)

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    final_labels,
    final_preds,
    labels=list(range(num_classes)),
    zero_division=0
)

cm = confusion_matrix(final_labels, final_preds, labels=list(range(num_classes)))

class_accuracy = cm.diagonal() / cm.sum(axis=1)

class_metrics_df = pd.DataFrame({
    "Class": class_names,
    "Accuracy": class_accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Support": support
})

display(class_metrics_df)

class_metrics_path = RESULTS_DIR / "kmu_baseline_classwise_metrics.csv"
class_metrics_df.to_csv(class_metrics_path, index=False)

print("Class-wise metrics saved to:", class_metrics_path)

In [ ]:
metrics_plot_path = RESULTS_DIR / "kmu_baseline_accuracy_precision_recall.png"

x = np.arange(len(class_names))
width = 0.25

plt.figure(figsize=(10, 6))

plt.bar(x - width, class_metrics_df["Accuracy"], width, label="Accuracy")
plt.bar(x, class_metrics_df["Precision"], width, label="Precision")
plt.bar(x + width, class_metrics_df["Recall"], width, label="Recall")

plt.xticks(x, class_names, rotation=45)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("KMU-FED Baseline Class-wise Metrics")
plt.legend()

plt.tight_layout()
plt.savefig(metrics_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Metrics plot saved to:", metrics_plot_path)

In [ ]:
summary = {
    "model": "SqueezeNet 1.1",
    "train_dataset": "KMU-FED cropped train",
    "test_dataset": "KMU-FED cropped test",
    "num_classes": num_classes,
    "epochs": num_epochs,
    "batch_size": batch_size,
    "learning_rate": 0.0001,
    "best_test_accuracy": best_test_accuracy,
    "final_test_accuracy": final_test_acc,
    "final_test_loss": final_test_loss,
    "macro_precision": overall_precision,
    "macro_recall": overall_recall,
    "model_path": str(best_model_path)
}

summary_df = pd.DataFrame([summary])

summary_csv_path = RESULTS_DIR / "kmu_baseline_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

display(summary_df)

print("Summary saved to:", summary_csv_path)

summary_df = pd.DataFrame([summary])

summary_csv_path = RESULTS_DIR / "kmu_baseline_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

display(summary_df)

print("Summary saved to:", summary_csv_path)

## Final Summary

SqueezeNet 1.1 was trained on the cropped KMU-FED training set and evaluated on the cropped KMU-FED test set.

This result is the baseline performance for the source dataset.

The saved outputs are:

- best trained model: `squeezenet_kmu_fed_best.pth`
- training history: `kmu_baseline_training_history.csv`
- classification report: `kmu_baseline_classification_report.txt`
- confusion matrix: `kmu_baseline_confusion_matrix.csv`
- summary file: `kmu_baseline_summary.csv`

The next step is `04_cross_dataset_testing.ipynb`, where the same saved KMU-FED trained model will be tested on KDEF and FER2013 without retraining.